In [33]:
# Lib imports
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import numpy as np


# Enable eager execution (required for .numpy() calls)
tf.config.run_functions_eagerly(True)


In [34]:

# DATASET DIRECTORY CONFIGURATION
# Download and unzip the dataset from Kaggle, set the directory paths accordingly.
train_dir = "C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/train"  # e.g. './muffin-vs-chihuahua/train'
test_dir = "C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test"    # e.g. './muffin-vs-chihuahua/test'

In [35]:
# IMAGE PARAMETERS
# Used to resize the input images, also will determine the input size of your input layer.
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

In [36]:
import os
os.path.exists(test_dir)

True

In [37]:
import os

folder = "C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test/chihuahua"
print(os.listdir(folder))

['img_0_1071.jpg', 'img_0_1074.jpg', 'img_0_1082.jpg', 'img_0_1087.jpg', 'img_0_11.jpg', 'img_0_1107.jpg', 'img_0_1110.jpg', 'img_0_1112.jpg', 'img_0_1117.jpg', 'img_0_1122.jpg', 'img_0_1133.jpg', 'img_0_1139.jpg', 'img_0_1150.jpg', 'img_0_1151.jpg', 'img_0_1154.jpg', 'img_0_1159.jpg', 'img_0_1161.jpg', 'img_0_1169.jpg', 'img_0_1172.jpg', 'img_0_1180.jpg', 'img_0_1191.jpg', 'img_0_1196.jpg', 'img_0_1200.jpg', 'img_0_1201.jpg', 'img_0_1212.jpg', 'img_0_137.jpg', 'img_0_138.jpg', 'img_0_146.jpg', 'img_0_151.jpg', 'img_0_152.jpg', 'img_0_16.jpg', 'img_0_17.jpg', 'img_0_177.jpg', 'img_0_18.jpg', 'img_0_184.jpg', 'img_0_203.jpg', 'img_0_22.jpg', 'img_0_229.jpg', 'img_0_235.jpg', 'img_0_24.jpg', 'img_0_274.jpg', 'img_0_275.jpg', 'img_0_290.jpg', 'img_0_292.jpg', 'img_0_295.jpg', 'img_0_298.jpg', 'img_0_30.jpg', 'img_0_301.jpg', 'img_0_302.jpg', 'img_0_329.jpg', 'img_0_337.jpg', 'img_0_340.jpg', 'img_0_343.jpg', 'img_0_345.jpg', 'img_0_346.jpg', 'img_0_352.jpg', 'img_0_368.jpg', 'img_0_383.jp

In [38]:
# DATA PREPROCESSING & AUGMENTATION
# Optional but recommended for image processing tasks, especially with limited data.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.
Found 1184 images belonging to 2 classes.


In [39]:
import os

root = train_dir  # or r"C:\path\to\train_1"
bad_files = []

valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".gif")

for r, d, f_list in os.walk(root):
    for f in f_list:
        if not f.lower().endswith(valid_ext):
            bad_files.append(os.path.join(r, f))

bad_files

[]

In [40]:
# SIMPLE CNN MODEL ARCHITECTURE

# Some modifications are applied
initial_learning_rate = 0.001
# We are combining ExponentialDecay with Adam optimizer for better learning rate management
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=10000,
    decay_rate=0.9,
    staircase=True
)

# Create the optimizer with the learning rate schedule
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Applied dropout layers to reduce overfitting
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.MaxPooling2D(2, 2),
    # layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    # layers.Dropout(0.25),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    # layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

In [41]:

# Configure the model optimizers, loss function, and metrics
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) # old
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [42]:
# TRAINING THE CNN
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 51s 426ms/step - accuracy: 0.7434 - loss: 0.5364 - val_accuracy: 0.7714 - val_loss: 0.4917
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 77s 646ms/step - accuracy: 0.8263 - loss: 0.3924 - val_accuracy: 0.8201 - val_loss: 0.4218
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 79s 663ms/step - accuracy: 0.8556 - loss: 0.3295 - val_accuracy: 0.8709 - val_loss: 0.2988
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 80s 668ms/step - accuracy: 0.8677 - loss: 0.3166 - val_accuracy: 0.9143 - val_loss: 0.2531
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 79s 662ms/step - accuracy: 0.8799 - loss: 0.2873 - val_accuracy: 0.8921 - val_loss: 0.2548
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 43s 329ms/step - accuracy: 0.8794 - loss: 0.2827 - val_accuracy: 0.9069 - val_loss: 0.2396
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 37s 308ms/step - accuracy: 0.8897 - loss: 0.2720 - val_accuracy: 0.9132 - val_loss: 0.2347
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 37s 311ms/step - accuracy: 0.8936 - loss: 0

In [43]:
# EVALUATE THE MODEL
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 152ms/step - accuracy: 0.9164 - loss: 0.1996
Test Accuracy: 0.9163851141929626


In [44]:

# SAVE THE MODEL
model.save('muffin_vs_chihuahua_cnn.h5')

In [45]:
# SIMPLE INFERENCE SCRIPT
from tensorflow.keras.preprocessing import image

def predict_image(img_path, model_path='muffin_vs_chihuahua_cnn.h5'):
    model = tf.keras.models.load_model(model_path)
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred = model.predict(img_array)[0,0]
    label = "Chihuahua" if pred >= 0.5 else "Muffin"
    print(f"Prediction: {label} (confidence: {pred:.2f})")

In [46]:
# Example usage:
predict_image("C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test/chihuahua/img_0_5.jpg")
predict_image("C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test/muffin/img_0_0.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


Prediction: Muffin (confidence: 0.31)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction: Chihuahua (confidence: 0.94)


In [47]:
from tensorflow.keras import regularizers

# L2 regularization factor
l2_factor = 0.001

model_improved = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_factor),
                  input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_factor)),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_factor)),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_factor)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile with the same optimizer and learning rate schedule
model_improved.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


In [48]:
# Create a new optimizer instance with the same learning rate schedule
optimizer_improved = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Compile the improved model with the new optimizer
model_improved.compile(optimizer=optimizer_improved, 
                       loss='binary_crossentropy', 
                       metrics=['accuracy'])

In [49]:
history_improved = model_improved.fit(
    train_generator,
    epochs=10,  # you can increase if needed
    validation_data=val_generator
)


Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 37s 309ms/step - accuracy: 0.6212 - loss: 0.9031 - val_accuracy: 0.7175 - val_loss: 0.7258
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 39s 326ms/step - accuracy: 0.7785 - loss: 0.6190 - val_accuracy: 0.8212 - val_loss: 0.5940
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 43s 358ms/step - accuracy: 0.8141 - loss: 0.5244 - val_accuracy: 0.8265 - val_loss: 0.5012
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 42s 356ms/step - accuracy: 0.8115 - loss: 0.5067 - val_accuracy: 0.8656 - val_loss: 0.4104
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 42s 351ms/step - accuracy: 0.8450 - loss: 0.4610 - val_accuracy: 0.8741 - val_loss: 0.3774
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 42s 353ms/step - accuracy: 0.8390 - loss: 0.4366 - val_accuracy: 0.8635 - val_loss: 0.4187
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 40s 339ms/step - accuracy: 0.8537 - loss: 0.4148 - val_accuracy: 0.9016 - val_loss: 0.3104
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 37s 307ms/step - accuracy: 0.8527 - loss: 0

In [50]:
test_loss_improved, test_acc_improved = model_improved.evaluate(test_generator)
print(f"Improved CNN Test Accuracy: {test_acc_improved:.4f}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 151ms/step - accuracy: 0.8666 - loss: 0.3759
Improved CNN Test Accuracy: 0.8666


In [51]:
def predict_image_improved(img_path, model=model_improved):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred = model.predict(img_array)[0,0]
    label = "Chihuahua" if pred >= 0.5 else "Muffin"
    print(f"Prediction: {label} (confidence: {pred:.2f})")

# Run predictions
predict_image_improved("C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test/chihuahua/img_0_5.jpg")
predict_image_improved("C:/College/3rd-Year/codes/Datasets-ANN/Muffin-Chihuahua/test/muffin/img_0_0.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction: Muffin (confidence: 0.16)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Prediction: Chihuahua (confidence: 0.58)


In [52]:
model_improved.save("exercise_6_trained_model_improved.h5")
